In [0]:
# Configuration
from pyspark.sql import functions as F, Window

dbutils.widgets.combobox("catalog", "workspace", ["workspace", "dbr_dev"], "Unity Catalog")
dbutils.widgets.combobox("bronze_schema", "bronze", ["bronze", "gabrielajaniszews786_bronze"], "Bronze schema")
dbutils.widgets.combobox("silver_schema", "silver", ["silver", "gabrielajaniszews786_silver"], "Silver schema")
dbutils.widgets.dropdown("run_checks", "true", ["true", "false"], "Verification / experimentation")
RUN_CHECKS = dbutils.widgets.get("run_checks") == "true"
CATALOG       = dbutils.widgets.get("catalog")
BRONZE_SCHEMA = dbutils.widgets.get("bronze_schema")
SILVER_SCHEMA = dbutils.widgets.get("silver_schema")

BRONZE_SENSOR = f"{CATALOG}.{BRONZE_SCHEMA}.sensor_data"
VALID_SILVER_SENSOR = f"{CATALOG}.{SILVER_SCHEMA}.valid_sensor"
CHECKED_SILVER_SENSOR = f"{CATALOG}.{SILVER_SCHEMA}.checked_sensor"
QUARANTINE_SILVER_SENSOR = f"{CATALOG}.{SILVER_SCHEMA}.quarantine_sensor"




In [0]:
display(spark.sql(f"SELECT * FROM {BRONZE_SENSOR} LIMIT 10"))

In [0]:
source_df = spark.sql(f"SELECT * FROM {BRONZE_SENSOR}")
display(source_df.count())


In [0]:
checked_df = spark.sql(f"SELECT * FROM {CHECKED_SILVER_SENSOR}")
display(checked_df.count())
valid_df = spark.sql(f"SELECT * FROM {VALID_SILVER_SENSOR}")
display(valid_df.count())
quarantine_df = spark.sql(f"SELECT * FROM {QUARANTINE_SILVER_SENSOR}")
display(quarantine_df.count())

In [0]:
def reconcile(bronze_df, valid_df, quarantine_df, tolerance=0):
    bronze_count = bronze_df.count()
    valid_count = valid_df.count()
    quarantine_count = quarantine_df.count()
    gap = bronze_count - (valid_count + quarantine_count)
    assert gap == 0, f"Gap between bronze and silver: {gap:,}"
    assert bronze_count == valid_count + quarantine_count, f"Bronze count does not match valid + quarantine: {bronze_count:,} != {valid_count:,} + {quarantine_count:,}"
